# Emotional Sinhala Speech Dataset — Kaggle pilot

Builds an emotion-labeled Sinhala speech dataset from an SLBC radio drama
(*Muwan Palassa*) on the Internet Archive. Designed to be run with **Run All**.

**Manifest-first:** the output is `manifest.csv` + `report.json`, **not** redistributed
audio. The source items carry no license field — treat them as copyrighted.

---

## Do these 3 things BEFORE Run All

**1. Settings panel (right):**

| Setting | Value |
|---|---|
| Accelerator | **GPU T4 ×2** |
| Internet | **ON** |
| Persistence | **Files only** (keeps `/kaggle/working` between sessions) |

**2. Add-ons → Secrets:** add `HF_TOKEN` = a HuggingFace **classic token with `Read` scope**,
and make sure it is **attached** to this notebook.

**3. Accept the gated pyannote terms** — logged into HF with the *same account that issued
that token* (your Kaggle email is irrelevant; the two services are unrelated):

- <https://huggingface.co/pyannote/speaker-diarization-3.1>
- <https://huggingface.co/pyannote/segmentation-3.0>
- <https://huggingface.co/pyannote/speaker-diarization-community-1>

Each asks for *Company/university* and *Website* (both required — e.g. `University of
Moratuwa` / `https://uom.lk`), then **Agree and access repository**. Approval is instant.

> Cell 4 checks all three and picks a working diarization model automatically. If none are
> accessible the run still completes — speakers just fall back to `SPEAKER_UNK`, which you
> can backfill later. Nothing here hard-fails on gating.

**Runtime:** ~5 min for the smoke check, then ~20–40 min for the full episode.

In [ ]:
# === 1. CONFIG — the only cell you normally edit ==============================
IDENTIFIER = "muwan-palassa-140113"   # archive.org item to pilot on
WORK       = "/kaggle/working/eesd"   # output root

RUN_SMOKE  = True    # ~2 min wiring check on the first 3 minutes (tiny ASR model)
RUN_FULL   = True    # ~20-40 min: the real pilot, full episode, whisper large-v3
FORCE      = True    # ignore cached per-stage state (needed after any code change)

REPO = "https://github.com/DSEgrp18/Dataset-creation-withEmotion.git"
print("identifier:", IDENTIFIER, "| work:", WORK,
      "| smoke:", RUN_SMOKE, "| full:", RUN_FULL)

In [ ]:
# === 2. environment + HuggingFace token ======================================
import os, subprocess, sys
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # Kaggle gives T4 x2; pin to one GPU

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("OK: HF token loaded from Kaggle secret.")
except Exception as e:
    print("!! No HF_TOKEN secret:", e)
    print("!! The run will continue, but diarization will fall back to SPEAKER_UNK.")

def run(args):
    """Run the pipeline, streaming its output into the notebook."""
    print(">>", " ".join(str(a) for a in args), flush=True)
    return subprocess.run([str(a) for a in args]).returncode

In [ ]:
# === 3. get / update the pipeline code =======================================
%env GIT_TERMINAL_PROMPT=0
# ^ makes git fail fast instead of hanging on a hidden username prompt
#   (that prompt means the repo is private -> make it public, or use a PAT below)
import os
os.chdir("/kaggle/working")
if not os.path.isdir("Dataset-creation-withEmotion"):
    !git clone $REPO
os.chdir("/kaggle/working/Dataset-creation-withEmotion")
!git pull --ff-only
!git log --oneline -1

# private repo? store a PAT as the Kaggle secret GITHUB_TOKEN and use:
# tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
# !git clone https://{tok}@github.com/DSEgrp18/Dataset-creation-withEmotion.git

In [ ]:
# === 4. install deps (~2-3 min) ==============================================
# torch / torchaudio / numpy / scipy ship preinstalled on Kaggle with CUDA wheels.
# Do NOT reinstall them -- everything below is additive.
!pip install -q internetarchive librosa soundfile pyloudnorm faster-whisper \
    "pyannote.audio>=3.1" demucs transformers
print("deps installed")

In [ ]:
# === 5. probe HuggingFace access -> choose a diarization model ================
# Diarization pulls weights from several gated repos. Rather than guess, probe
# each one and pick a pipeline we can actually load.
import os
D31       = "pyannote/speaker-diarization-3.1"
SEG30     = "pyannote/segmentation-3.0"
COMMUNITY = "pyannote/speaker-diarization-community-1"

DIAR_MODEL = "auto"
tok = os.environ.get("HF_TOKEN")
if not tok:
    print("!! no HF_TOKEN -> diarization will be skipped (SPEAKER_UNK)")
else:
    from huggingface_hub import HfApi
    api = HfApi()
    try:
        print("token belongs to HF account:", api.whoami(token=tok)["name"])
    except Exception as e:
        print("!! token rejected by HuggingFace:", str(e)[:120])
    access = {}
    for r in (D31, SEG30, COMMUNITY):
        try:
            api.model_info(r, token=tok); access[r] = True
        except Exception as e:
            access[r] = False
            print("   reason:", str(e)[:100])
        print("   OK     " if access[r] else "   BLOCKED", r)

    if all(access.values()):
        DIAR_MODEL = D31
    elif access.get(COMMUNITY):
        DIAR_MODEL = COMMUNITY      # self-contained; sidesteps 3.1's extra gates
    else:
        DIAR_MODEL = "auto"
        print()
        print("!! No diarization model is accessible.")
        print("!! The pipeline will still run; speakers become SPEAKER_UNK.")
        print("!! Accept the terms on the 3 URLs at the top, then re-run this notebook.")

print()
print("=> using --diarization-model", DIAR_MODEL)

In [ ]:
# === 6. smoke: all 9 stages on the first ~3 min ==============================
# Purpose: prove nothing CRASHES. Uses a tiny ASR model, so the Sinhala text is
# poor and usable yield is often 0 -- do NOT judge quality from this run.
if RUN_SMOKE:
    args = ["python", "build_emotional_sinhala_dataset.py", "--stage", "all",
            "--smoke", "--identifiers", IDENTIFIER, "--work-dir", WORK,
            "--diarization-model", DIAR_MODEL]
    if FORCE:
        args.append("--force")
    rc = run(args)
    print("\nsmoke exit code:", rc, "(0 = ok)")
else:
    print("skipped (RUN_SMOKE = False)")

In [ ]:
# === 7. verify diarization actually ran ======================================
# This failure is silent by design (the pipeline continues), so check explicitly.
import json, glob
speakers, turns = set(), 0
for p in glob.glob(f"{WORK}/meta/*.json"):
    meta = json.load(open(p))
    for sf in meta.get("source_files", []):
        for t in sf.get("diarization", []):
            speakers.add(t["speaker"]); turns += 1

print("speakers:", sorted(speakers), "| turns:", turns)
print()
if not speakers or speakers == {"SPEAKER_UNK"}:
    print("!! Diarization did not run -- clips are NOT speaker-pure.")
    print("!! Not fatal: yield and emotion numbers below are still valid.")
    print("!! Fix access (cell 5 output), then backfill later with:")
    print(f"!!   --stage diarize --force --work-dir {WORK}")
else:
    print(f"OK: {len(speakers)} distinct speakers across {turns} turns.")

In [ ]:
# === 8. THE REAL PILOT: full episode, whisper large-v3 (~20-40 min) ==========
if RUN_FULL:
    args = ["python", "build_emotional_sinhala_dataset.py", "--stage", "all",
            "--identifiers", IDENTIFIER, "--work-dir", WORK,
            "--diarization-model", DIAR_MODEL]
    if FORCE:
        args.append("--force")
    rc = run(args)
    print("\nfull run exit code:", rc, "(0 = ok)")
else:
    print("skipped (RUN_FULL = False)")

In [ ]:
# === 9. yield funnel + emotion distribution ==================================
import json, os
import pandas as pd

rep = json.load(open(f"{WORK}/report.json"))
print("FUNNEL (minutes):", json.dumps(rep.get("funnel_minutes", {}), indent=2))
print("USABLE YIELD %:", rep.get("yield_percent"))
print("COUNTS:", rep.get("counts", {}))

ed = rep.get("emotion_distribution", {})
cc = ed.get("category_counts", {})
if cc:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
    ax[0].bar(list(cc.keys()), list(cc.values()))
    ax[0].set_title("Emotion distribution (usable clips)")
    ax[0].tick_params(axis="x", rotation=30)
    ah = ed.get("arousal_hist", {})
    if ah:
        ax[1].bar(list(ah.keys()), list(ah.values()))
        ax[1].set_title("Arousal histogram")
        ax[1].tick_params(axis="x", rotation=60)
    plt.tight_layout(); plt.show()
else:
    print("\nNo usable-clip emotion labels (normal for a --smoke-only run).")

for name in ("manifest.csv", "needs_manual_transcription.csv"):
    p = f"{WORK}/{name}"
    if os.path.exists(p):
        df = pd.read_csv(p)
        print(f"\n=== {name}: {len(df)} rows ===")
        if len(df):
            print(df.head(10).to_string())

In [ ]:
# === 10. listen before you trust it ==========================================
import os, pandas as pd
from IPython.display import Audio, display

p = f"{WORK}/manifest.csv"
df = pd.read_csv(p) if os.path.exists(p) else pd.DataFrame()
if len(df):
    for _, r in df.head(5).iterrows():
        print(f"[{r.emotion_label}] arousal={r.arousal} valence={r.valence} "
              f"asr={r.asr_conf} spk={r.speaker_id}")
        print("   ", r.text)
        display(Audio(f"{WORK}/clips/{r.clip_id}.wav"))
else:
    print("No usable clips yet -- check needs_manual_transcription.csv instead:")
    p2 = f"{WORK}/needs_manual_transcription.csv"
    if os.path.exists(p2):
        dfm = pd.read_csv(p2)
        for _, r in dfm.head(5).iterrows():
            print(f"[{r.reason}] arousal={r.arousal} asr={r.asr_conf}")
            display(Audio(f"{WORK}/clips/{r.clip_id}.wav"))

In [ ]:
# === 11. SAVE the dataset before the session ends ============================
# /kaggle/working is wiped between sessions unless Persistence = 'Files only'.
# These files ARE the dataset (manifest-first) -- download them from the
# right-hand Output panel and commit them. Never commit the audio.
import shutil, os
for name in ("manifest.csv", "needs_manual_transcription.csv",
             "report.json", "download_manifest.csv"):
    src = f"{WORK}/{name}"
    if os.path.exists(src):
        shutil.copy(src, f"/kaggle/working/{name}")
        print(f"ready to download: {name} ({os.path.getsize(src)} bytes)")

## How to read your results

- **`yield_percent`** — expect **<10–20 %**; 5–15 % is normal and healthy for radio drama.
  ~0 % after the *full* run (not the smoke run) means something is misconfigured.
- **`funnel_minutes`** — shows where audio was lost: separate → diarize → VAD → filter.
- **`emotion_distribution`** — if it is nearly all `neutral`/`calm_content`, the expressive
  clips are being lost, almost always to ASR failure on shouted/cried speech.
- **`needs_manual_transcription.csv`** — high-arousal clips that failed ASR. **These are the
  most valuable clips for expressive TTS.** They are kept for a human transcription pass
  rather than discarded. Budget time for this.

**If yield is too low**, loosen the filters instead of accepting the loss — add to the
`args` list in cell 8:

```python
"--min-asr-conf", "0.4", "--min-align-conf", "0.4", "--min-snr-db", "5"
```

## Scaling up

Only after reviewing the numbers above. The pipeline is idempotent, so processed episodes
are skipped and you can add identifiers across sessions — set in cell 1:

```python
IDENTIFIER = "muwan-palassa-210113,MuwanPalassa29816,muwanpalassa_27513"
```

Watch the 30 h GPU quota: roughly 20–40 min per 25-minute episode.